# Exploratory analysis

Explore the cleaned Milestone 1 data. The checked-in file is only a 9-row sample; use the full authorized dataset before making statistical or modeling conclusions. The prediction target is the next observed `global_active_power` value (kW), not necessarily exactly one minute ahead.

In [ ]:
from pathlib import Path
import sys

source_dirs = [Path.cwd() / 'src', Path.cwd().parent / 'src']
source_dir = next(path for path in source_dirs if (path / 'ml' / 'data_loader.py').is_file())
sys.path.insert(0, str(source_dir))
from ml.data_loader import load_processed_data, PROJECT_ROOT

data = load_processed_data()
print('Shape:', data.shape)
print('Columns:', data.columns.tolist())
print('Dtypes:\n', data.dtypes.to_string())
print('Missing values:\n', data.isna().sum().to_string())
print('Duplicate rows:', int(data.drop(columns='timestamp').duplicated().sum()))
display(data.head())
display(data.describe(include='all').T)
timestamp_gaps = data['timestamp'].diff().dropna()
print('Timestamp range:', data['timestamp'].min(), 'to', data['timestamp'].max())
print('Observed timestamp intervals:\n', timestamp_gaps.value_counts().to_string())
print('Target summary (kW):\n', data['global_active_power'].describe().to_string())

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

figures = PROJECT_ROOT / 'reports' / 'figures'
figures.mkdir(parents=True, exist_ok=True)
numeric = data.select_dtypes(include='number')

fig, ax = plt.subplots(figsize=(8, 4))
sns.histplot(data=data, x='global_active_power', kde=False, ax=ax)
ax.set(title='Global active power distribution', xlabel='Power (kW)', ylabel='Observation count')
fig.tight_layout()
fig.savefig(figures / 'active_power_distribution.png', dpi=150)
plt.show()

fig, ax = plt.subplots(figsize=(10, 4))
sns.lineplot(data=data, x='timestamp', y='global_active_power', marker='o', ax=ax)
ax.set(title='Global active power over time', xlabel='Timestamp', ylabel='Power (kW)')
fig.tight_layout()
fig.savefig(figures / 'active_power_over_time.png', dpi=150)
plt.show()

fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(numeric.corr(), annot=True, fmt='.2f', cmap='vlag', center=0, ax=ax)
ax.set(title='Numeric measurement correlation (descriptive only)')
fig.tight_layout()
fig.savefig(figures / 'numeric_correlation.png', dpi=150)
plt.show()

## Interpretation and limitations

The notebook reports quality checks and saves charts without removing rows or changing measurements. Correlations are descriptive and do not establish useful predictive features. The local sample has only nine complete rows and contains a two-minute gap; its plots and statistics are not representative evidence for the full dataset.